# Weighted Deduplication Workflow

> Library-first example for running the production deduplication pipeline on a new reference CSV.

This notebook follows the same workflow documented in `DEDUPLICATION_WORKFLOW.md`:
1. Load records from CSV.
2. Build blocked candidate pairs.
3. Validate records into a typed cache.
4. Score pairs with the weighted model.
5. Resolve predicted duplicate clusters and export the deduplicated file.

## 1. Load libraries and modules

In [ ]:
import sys
from pathlib import Path

import pandas as pd

sys.path.insert(0, "..")

from app.candidate_selection import build_blocked_pairs
from app.data_models import Paper
from app.dedupe import Deduper
from app.import_references import CsvLoadConfig, DEFAULT_COLUMNS, load_reference_csv
from app.record_cache import build_record_cache
from app.record_resolution import remove_duplicates

REPO_DIR = Path.cwd().parent
INPUT_CSV = REPO_DIR / "notebooks/data/diabetes_data.csv"

# Probability threshold for duplicate decisions
THRESHOLD = 0.85

## 2. Load reference records

Required for this notebook path:
- `recordid` (unique record identifier)

Common bibliographic fields used by scoring and retention strategies include:
- `title`
- `authors`
- `year`
- `journal`
- `doi`
- `pages`
- `issue`
- `volume`
- `abstract`

In [5]:
df = load_reference_csv(
    INPUT_CSV,
    CsvLoadConfig(columns=DEFAULT_COLUMNS,
))

print(f"Loaded {len(df):,} reference records")

df.head()

Loaded 1,845 reference records


,authors,year,journal,doi,title,pages,volume,issue,abstract,recordid
0,Adeli K.Lewis G. F.,2008,Curr Opin Lipidol,10.1097/MOL.0b013e3282ffaf82,Intestinal lipoprotein overproduction in insul...,221-8,19,3,PURPOSE OF REVIEW: Excessive postprandial lipe...,1
1,Adeli K.Lewis G. F.,2008,Curr Opin Lipidol,http://dx.doi.org/10.1097/MOL.0b013e3282ffaf82,Intestinal lipoprotein overproduction in insul...,221-228,19,3,PURPOSE OF REVIEW: Excessive postprandial lipe...,699
2,Adeli K.Lewis G. F.,2008,Curr Opin Lipidol,10.1097/MOL.0b013e3282ffaf82,Intestinal lipoprotein overproduction in insul...,221-8,19,3,PURPOSE OF REVIEW: Excessive postprandial lipe...,728
3,Adeli K.Lewis G. F.,2008,Curr Opin Lipidol,http://dx.doi.org/10.1097/MOL.0b013e3282ffaf82,Intestinal lipoprotein overproduction in insul...,221-228,19,3,PURPOSE OF REVIEW: Excessive postprandial lipe...,1447
4,Adingupu D. D.Gronros J.Behrendt M.Westergren ...,2017,European Heart Journal,NaN,SGLT2 inhibition improves coronary microvascul...,893,38,NaN,Background: Trea]t with SGLT2i have been sugge...,1630


## 3. Generate candidate pairs using blocking

Comparing every record against every other record is expensive.

Blocking creates a smaller set of plausible candidate pairs using shared characteristics (for example title fragments, author names, publication details, etc.).

In [6]:
pairs_df = build_blocked_pairs(df, dup_column=None)

print(f"Generated {len(pairs_df):,} candidate pairs")

pairs_df.head()

Generated 8,487 candidate pairs


c:\Users\qtnzkh4\OneDrive - University College London\deduplication-toolkit-now\notebooks\..\app\normalisers.py:65: FutureWarning: Possible set difference at position 2
  page_range = re.sub(r"[---]+", "-", page_range).strip()


,id_a,id_b
0,290,1031
1,188,927
2,9,737
3,200,939
4,116,852


## 4. Convert records into Paper objects

The deduplication model operates on Paper objects rather than raw dataframe rows.

In [7]:
from loguru import logger
logger.remove()

candidate_ids = set(pairs_df["id_a"]).union(
    pairs_df["id_b"]
)
all_record_ids = set(df["recordid"].dropna().astype(int))
excluded_ids = sorted(all_record_ids - candidate_ids)

record_cache, validation_errors, invalid_records = build_record_cache(
    df,
    id_column="recordid",
    include_ids=candidate_ids,
    record_model=Paper,
    return_invalid_records=True,
)

print(f"Created {len(record_cache):,} Paper objects")
print(
    f"Excluded {len(excluded_ids):,} records because they did not appear in any candidate pair"
 )

excluded_records_df = df.loc(
    df["recordid"].isin(excluded_ids),
    ["recordid", "title", "year", "journal"],
]

if validation_errors:
    print(f"Skipped {validation_errors} invalid records")
    title_lookup = df.set_index("recordid")["title"].to_dict()
    invalid_records_df = pd.DataFrame(
        [
            {
                "recordid": item["recordid"],
                "title": title_lookup.get(item["recordid"]),
                "error": "; ".join(error["msg"] for error in item["errors"]),
            }
            for item in invalid_records
        ]
    )
else:
    invalid_records_df = pd.DataFrame(columns=["recordid", "title", "error"])

Created 1,838 Paper objects
Excluded 7 records because they did not appear in any candidate pair


## 5. Initialize the deduper

The weighted logistic model is accessed through `Deduper`.
A single example record is used only to initialize the object; scoring happens pair-by-pair in the next step.

In [8]:
example_record = next(iter(record_cache.values()))

deduper = Deduper(
    reference=example_record,
    candidates=[example_record],
)

## 6. Score candidate pairs

> Each blocked pair is scored by the weighted deduper and converted into a probability.

Output columns created in this step:
- `id_a`, `id_b`
- `probability`
- `duplicate_prediction` (boolean threshold decision)

In [9]:
results = []

for row in pairs_df.itertuples(index=False):

    record_a = record_cache.get(row.id_a)
    record_b = record_cache.get(row.id_b)

    if record_a is None or record_b is None:
        continue

    probability = deduper.dedupe_weighted(
        record_a,
        record_b,
    )

    results.append(
        {
            "id_a": row.id_a,
            "id_b": row.id_b,
            "probability": probability,
            "duplicate_prediction": probability >= THRESHOLD,
        }
    )


results_df = pd.DataFrame(results)

results_df.head()

,id_a,id_b,probability,duplicate_prediction
0,290,1031,0.362927,False
1,188,927,0.999997,True
2,9,737,0.999376,True
3,200,939,0.999997,True
4,116,852,0.999376,True


## 7. Remove duplicates at record level (cluster-aware)

Predicted duplicate pairs are converted into connected components (duplicate clusters).
Each cluster keeps one canonical record and removes the rest.

Retention strategies:
- `min_recordid`: deterministic baseline (smallest `recordid`)
- `metadata_richness`: keep the row with the most populated bibliographic fields
- `prefer_doi_abstract`: prioritize DOI + abstract presence, then metadata richness

Optional enrichment can backfill missing values in the kept row from other records in the same cluster.

In [10]:
RETENTION_STRATEGY = "prefer_doi_abstract"
ENRICH_KEPT_RECORDS = True

deduplicated_df, removed_duplicates_df, decisions_df = remove_duplicates(
    df_records=df,
    scored_pairs=results_df,
    threshold=THRESHOLD,
    strategy=RETENTION_STRATEGY,
    enrich_kept_records=ENRICH_KEPT_RECORDS,
    probability_column="probability",
 )

print(f"Original records: {len(df):,}")
print(f"Deduplicated records kept: {len(deduplicated_df):,}")
print(f"Records removed as duplicates: {len(removed_duplicates_df):,}")
print(f"Clusters with duplicates: {(decisions_df['cluster_size'] > 1).sum():,} records involved")

display(
    decisions_df.loc[decisions_df["cluster_size"] > 1, [
        "recordid",
        "predicted_cluster",
        "cluster_size",
        "keep",
        "kept_recordid",
    ]].head(20)
 )

deduplicated_df.head()

Original records: 1,845
Deduplicated records kept: 593
Records removed as duplicates: 1,252
Clusters with duplicates: 1,816 records involved


,recordid,predicted_cluster,cluster_size,keep,kept_recordid
1073,145,344,8,False,488
1074,469,344,8,False,488
1076,491,344,8,False,488
279,493,98,8,False,494
1077,883,344,8,False,488
1071,1212,344,8,False,488
1072,1232,344,8,False,488
1078,1235,344,8,False,488
280,1237,98,8,False,494
276,1238,98,8,False,494


,authors,year,journal,doi,title,pages,volume,issue,abstract,recordid,predicted_cluster
0,Ahmed H. A.May D. W.Fagan S. C.Segar L.,2015,Pharmacotherapy,10.1002/phar.1547,Vascular protection with dipeptidyl peptidase-...,277-97,35,3,The dipeptidyl peptidase-IV (DPP-IV) inhibitor...,2,6
1,Akoumianakis I.Antoniades C.,2017,Vascul Pharmacol,10.1016/j.vph.2017.07.001,Dipeptidyl peptidase IV inhibitors as novel re...,01-Apr,96-98,NaN,Dipeptidyl peptidase IV (DPP-IV) has been reve...,4,11
2,Alonso N.Julian M. T.Puig-Domingo M.Vives-Pi M.,2012,Front Endocrinol (Lausanne),10.3389/fendo.2012.00112,Incretin hormones as immunomodulators of ather...,112,3,NaN,Atherosclerosis results from endothelial cell ...,6,15
3,Apryatin S. A.Mzhelskaya K. V.Trusov N. V.Bala...,2016,Vopr Pitan,NaN,[Comparative characteristics of in vivo models...,14-23,85,6,In vivo simulation of lipid disorders (hyperli...,9,17
4,Bahtiyar G.Pujals-Kury J.Sacerdote A.,2018,Curr Diab Rep,10.1007/s11892-018-1043-z,Cardiovascular Effects of Different GLP-1 Rece...,92,18,10,PURPOSE OF REVIEW: Glucagon-like peptide-1 rec...,13,50


## 8. Export unique records as CSV

> Define output path and write the deduplicated record table to disk.

In [11]:


OUTPUT_CSV = REPO_DIR / "notebooks/data/diabetes_data_deduplicated.csv"
deduplicated_df.to_csv(OUTPUT_CSV, index=False)

print(f"Exported de-duplicated references to: {OUTPUT_CSV}")

Exported de-duplicated references to: c:\Users\qtnzkh4\OneDrive - University College London\deduplication-toolkit-now\notebooks\data\diabetes_data_deduplicated.csv
